# Решение задачи классификации с помощью RF

Обучим случайный лес на двух типах признаков: кастомные признаки полученные в результате EDA и BoW, сравним полученные результаты с линейными моделями.

## Подготовка данных и импорты

In [1]:
import sys

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    !pip install optuna

    from google.colab import drive

    drive.mount('/content/drive')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 19.5 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
import optuna
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, make_scorer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
import polars as pl
from sklearn.model_selection import train_test_split

DATA_PATH = "/content/drive/MyDrive/data/cleaned_comments.csv" if IS_COLAB else "../data/interim/cleaned_comments.csv"
RANDOM_SEED = 42

optuna.logging.set_verbosity(optuna.logging.CRITICAL)

In [3]:
label_mapping = {
    "NORMAL": 0,
    "INSULT": 1,
    "THREAT": 2,
    "OBSCENITY": 3
}
data = (
    pl.read_csv(DATA_PATH, )
    .filter(pl.col("cleaned_comment").is_not_null())
    .select(pl.exclude("is_normal", ""))
    .with_columns(pl.col("label").replace(label_mapping).cast(dtype=int))
)
train, test = train_test_split(data, test_size=0.2, stratify=data["label"], random_state=RANDOM_SEED)

## RF на кастомных признаках

In [ ]:
features_train, label_train = train.select(pl.exclude("label", "comment", "cleaned_comment", "comment_without_punct")), train["label"]
features_test, label_test = test.select(pl.exclude("label", "comment", "cleaned_comment", "comment_without_punct")), test["label"]

### Подбор гиперпараметров

In [5]:
def rf_on_custom_features(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 700),
        "criterion": trial.suggest_categorical("criterion", ["gini", "entropy"]),
        "max_depth": trial.suggest_int("max_depth", 8, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 1000),
        "max_features": trial.suggest_float("max_features", 0.0, 1.0),
        "class_weight": "balanced_subsample",
    }
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
    model = RandomForestClassifier(**params)
    score = cross_val_score(model, features_train, label_train, cv=cv, scoring=make_scorer(f1_score, average="macro"))
    return score.mean()


study = optuna.create_study(direction="maximize")
study.optimize(rf_on_custom_features, n_trials=100, n_jobs=-1)
print(f"Best score: {study.best_value:.3f}")
print(f"Best params: {study.best_params}")

Best score: 0.421
Best params: {'n_estimators': 456, 'criterion': 'gini', 'max_depth': 18, 'min_samples_leaf': 1, 'max_features': 0.0069766108050558495}


### Обучение модели

In [6]:
custom_features_rf = RandomForestClassifier(
    n_estimators=456,
    criterion="gini",
    max_depth=18,
    min_samples_leaf=1,
    max_features=0.007,
    class_weight="balanced_subsample"
)
pred = custom_features_rf.fit(features_train, label_train).predict(features_test)

In [7]:
print(classification_report(label_test, pred))

              precision    recall  f1-score   support

           0       0.91      0.81      0.86     40552
           1       0.64      0.53      0.58      5713
           2       0.10      0.25      0.14      2356
           3       0.08      0.24      0.12       852

    accuracy                           0.74     49473
   macro avg       0.43      0.46      0.42     49473
weighted avg       0.83      0.74      0.78     49473



## RF на bag-of-words

In [8]:
vectors_train, label_train = train["cleaned_comment"], train["label"]
vectors_test, label_test = test["cleaned_comment"], test["label"]

### Подбор гиперпараметров

In [9]:
def rf_on_tfidf(trial):
    rf_params = {
        "n_estimators": trial.suggest_int("n_estimators", 250, 700),
        "criterion": trial.suggest_categorical("criterion", ["gini", "entropy"]),
        "max_depth": trial.suggest_int("max_depth", 8, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10-0),
        "max_features": trial.suggest_float("rf_max_features", 0.0, 1.0),
        "class_weight": "balanced_subsample",
    }
    max_features = None
    if trial.suggest_categorical("use_vectorizer_max_features", [True, False]):
        max_features = trial.suggest_int("vectorizer_max_features", 1e+4, 1e+6)
    pipe = Pipeline([
        ("vectorizer", TfidfVectorizer(max_features=max_features)),
        ("rf", RandomForestClassifier(**rf_params))
    ])
    pred = pipe.fit(vectors_train, label_train).predict(vectors_test)
    return f1_score(label_test, pred, average="macro")


study = optuna.create_study(direction="maximize")
study.optimize(rf_on_tfidf, n_trials=100, n_jobs=-1)
print(f"Best score: {study.best_value:.3f}")
print(f"Best params: {study.best_params}")

Best score: 0.650
Best params: {'n_estimators': 434, 'criterion': 'gini', 'max_depth': 18, 'min_samples_leaf': 7, 'rf_max_features': 0.0033082046170216867, 'use_vectorizer_max_features': True, 'vectorizer_max_features': 897925}


### Обучение модели

In [13]:
vectors_rf = Pipeline([
    ("vectorizer", TfidfVectorizer(max_features=897925)),
    ("rf", RandomForestClassifier(
        n_estimators=434,
        criterion="gini",
        max_depth=18,
        min_samples_leaf=7,
        max_features=0.003,
        class_weight="balanced_subsample"
    )),
])
pred = vectors_rf.fit(vectors_train, label_train).predict(vectors_test)

In [14]:
print(classification_report(label_test, pred))

              precision    recall  f1-score   support

           0       0.95      0.92      0.93     40552
           1       0.66      0.52      0.58      5713
           2       0.49      0.79      0.60      2356
           3       0.34      0.78      0.47       852

    accuracy                           0.86     49473
   macro avg       0.61      0.75      0.65     49473
weighted avg       0.88      0.86      0.87     49473



## Выводы

1. RF для кастомных признаков дает такое же качество как и линейные модели
2. RF для BoW работает хуже линейных моделей по нескольким причинам:
    * из-за большого количества признаков долго обучается и долго делает предсказание
    * качество классификации хуже из-за того что на BoW трудно выделить информативные пороговые признаки.

